# Computing Word-Level Reading Measures from Eye-Tracking Data

This tutorial demonstrates how to compute AOI-based word-level reading measures from eye-tracking data using predefined Areas of Interest (AOIs).

## What you will learn

In this tutorial, you will learn how to:

- load a DataFrame containing fixation events,
- load a DataFrame defining word-level AOIs with their bounding boxes,
- map fixations to the corresponding AOIs,
- compute word-level reading measures using `compute_reading_measures`, and
- inspect the resulting DataFrame of computed reading measures.

In [ ]:
from pathlib import Path
import polars as pl

from pymovements import Dataset
from pymovements import Events
from pymovements.stimulus.text import TextStimulus
from pymovements.measure.reading.processing import compute_reading_measures

We begin by loading a dataset containing fixation events together with the corresponding AOI definitions. In this tutorial, we use the `GGTG` dataset, which can be loaded as follows:

In [ ]:
dataset = Dataset('GGTG', path='data/GGTG')

# Download the dataset and extract all archives.
dataset.download()

# Load the dataset into memory for processing
dataset.load()

For simplicity, we restrict the analysis to a single subject and a single stimulus. Specifically, we use the first subject and the stimulus `goldfish-pos.text.0`. We then select only the required columns and add a column indicating the event type:

In [ ]:
stimulus = "goldfish-pos.text.0"
sample_fixation_path = dataset.paths.precomputed_events / dataset.fileinfo['precomputed_events']['filepath'][0]

fixations = pl.read_csv(sample_fixation_path)
fixations = fixations.filter(pl.col('stimulus') == stimulus)
fixations = fixations[['onset',
                       'offset',
                       'duration',
                       'location_x',
                       'location_y',
                      ]]
# add name column for the type of event
fixations = fixations.with_columns(name = pl.lit('fixation'))
fixations.head()

Next, we load the CSV file containing the AOI definitions for the selected stimulus:

In [ ]:
stimulus_rel_path = dataset.fileinfo['textstimulus'].filter(pl.col('stimulus') == stimulus).filter(pl.col('unit') == 'word')['filepath'][0]
aoi_path = dataset.paths.stimuli / stimulus_rel_path

aoi_df = pl.read_csv(aoi_path, separator=',')
aoi_df = aoi_df.with_columns(aoi_index = pl.col('index'))
aoi_df.head()

To use the AOI definitions, we convert the DataFrame into a `TextStimulus`:

In [ ]:
aoi_text_stimulus = TextStimulus(
                aoi_df,
                aoi_column='content',
                start_x_column='left',
                start_y_column='top',
                end_x_column='right',
                end_y_column='bottom',)
aoi_text_stimulus

Next, we map the fixation events to their corresponding AOIs. To do so, we create an `Events` DataFrame and use the `map_to_aois` function:

In [ ]:
events = Events(data=fixations)
events.map_to_aois(aoi_text_stimulus)
events.frame

With all required data prepared, we can now compute the reading measures using `compute_reading_measures`:

In [ ]:
rm_df = compute_reading_measures(
                fixations=events.frame,
                aois=aoi_df,
                word_index_column='aoi_index',
                word_column='content'
            )
rm_df.head()

The resulting DataFrame contains one row per word (Area of Interest, AOI) and subject, along with a range of eye-tracking reading measures.

#### Description of computed reading measures

| Column | Description |
|--------|-------------|
| `word` | The word (AOI) within the text. |
| `word_index` | Zero-based index of the word within the text. |
| `FFD` | **First Fixation Duration** — duration of the first fixation during the first pass only. |
| `SFD` | **Single Fixation Duration** — fixation duration when a word receives exactly one fixation; `0` if the word receives multiple fixations. |
| `FD` | **Fixation Duration** — duration of the first fixation on the word. |
| `FPRT` | **First Pass Reading Time** — sum of all fixation durations on the word during the first pass. |
| `FRT` | **First Reading Time** — total dwell time from first entering the word until first leaving it. |
| `TFT` | **Total Fixation Time** — total fixation time on the word (`FPRT + RRT`). |
| `RRT` | **Rereading Time** — sum of fixation durations occurring after the first pass. |
| `RPD_inc` | **Regression-Path Duration (inclusive)** — sum of all fixation durations from first entering the word until the first fixation to the right of the word, including fixations on the word itself. |
| `RPD_exc` | **Regression-Path Duration (exclusive)** — time spent on regressed words only, excluding fixations on the current word. |
| `RBRT` | **Right-Bounded Reading Time** — sum of fixation durations on the word before any word to its right is fixated. |
| `Fix` | **Fixation Indicator** — `1` if the word was fixated (`TFT > 0`). |
| `FPF` | **First Pass Fixation Indicator** — `1` if the word was fixated during the first pass (`FFD > 0`). |
| `RR` | **Rereading Indicator** — `1` if the word was reread (`RRT > 0`). |
| `FPReg` | **Regression Indicator** — `1` if regressions occurred (`RPD_exc > 0`). |
| `TRC_out` | **Total Regression Count (Outgoing)** — number of regressions originating from the word. |
| `TRC_in` | **Total Regression Count (Ingoing)** — number of regressions landing on the word. |
| `SL_in` | **Saccade Length (Ingoing)** — word distance between the current word and the previously fixated word at the time of the first fixation on the current word. |
| `SL_out` | **Saccade Length (Outgoing)** — word distance from the current word to the next fixated word, measured at the last fixation of the first reading pass. |
| `TFC` | **Total Fixation Count** — total number of fixations on the word. |
| `subject_id` | Identifier of the subject. |
| `text_id` | Identifier of the text. |